# Trout New Dataset EDA and Master Table

This notebook builds the first clean table for the new trout dataset on the HPC.

Goals:
- Confirm the expected HPC paths and files exist.
- Index image files from `cu`, `du`, `po`, `to`, and `tu` folders.
- Load `labeling.xlsx` and `demo.xlsx`.
- Merge image, label, length, and weight information into one `master_df`.
- Save audit tables that identify missing joins and within-fish age conflicts.

No model training is done in this notebook.

## 1. Setup

Run this first. The default paths are set for the HPC layout under `/home/jlc3q/data/Trout`. If needed, override them with environment variables before opening Jupyter.

In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

ROOT_DIR = Path(os.environ.get("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.environ.get("TROUT_CODE_DIR", str(ROOT_DIR / "code_new")))
DEMO_XLSX = Path(os.environ.get("TROUT_DEMO_XLSX", str(ROOT_DIR / "demo.xlsx")))
LABEL_XLSX = Path(os.environ.get("TROUT_LABEL_XLSX", str(ROOT_DIR / "labeling.xlsx")))

IMAGE_FOLDERS = ["cu", "du", "po", "to", "tu"]
OUTPUT_DIR = CODE_DIR / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 100
np.random.seed(SEED)


def normalize_text(value):
    if pd.isna(value):
        return np.nan
    return " ".join(str(value).strip().split())


def normalize_key(value):
    if pd.isna(value):
        return np.nan
    return normalize_text(value).lower()

print("ROOT_DIR:", ROOT_DIR)
print("CODE_DIR:", CODE_DIR)
print("DEMO_XLSX:", DEMO_XLSX, DEMO_XLSX.exists())
print("LABEL_XLSX:", LABEL_XLSX, LABEL_XLSX.exists())
for folder in IMAGE_FOLDERS:
    p = ROOT_DIR / folder
    print(f"{folder} exists:", p.exists(), p)

## 2. Excel Workbook Check

This checks the sheet names and column names before merging. It also verifies whether the `all` sheet in `demo.xlsx` agrees with the individual river/point sheets.

In [ ]:
demo_xls = pd.ExcelFile(DEMO_XLSX)
label_xls = pd.ExcelFile(LABEL_XLSX)

print("demo sheets:", demo_xls.sheet_names)
print("label sheets:", label_xls.sheet_names)

for sheet in demo_xls.sheet_names:
    cols = pd.read_excel(DEMO_XLSX, sheet_name=sheet, nrows=0).columns.tolist()
    print(f"demo::{sheet} columns:", cols)

for sheet in label_xls.sheet_names:
    cols = pd.read_excel(LABEL_XLSX, sheet_name=sheet, nrows=0).columns.tolist()
    print(f"label::{sheet} columns:", cols)

In [ ]:
def standardize_demo_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        "lenght (mm)": "length_mm",
        "length (mm)": "length_mm",
        "weight (g)": "weight_g",
    }
    out = df.rename(columns=rename_map).copy()
    return out


def fish_key_from_demo_id(value, river: str | None = None, point: int | None = None):
    if pd.isna(value):
        return np.nan

    numeric = pd.to_numeric(value, errors="coerce")
    if pd.notna(numeric) and river is not None and point is not None:
        return f"{river.lower()}{point}_{int(numeric):03d}"

    text = normalize_key(value)
    text = text.replace(" ", "_")
    match = re.match(r"^([a-z]+)_?(\d+)_?(\d+)$", text)
    if match:
        river_name, point_text, fish_text = match.groups()
        return f"{river_name}{int(point_text)}_{int(fish_text):03d}"
    return text


def build_demo_from_individual_sheets(xlsx_path: Path) -> pd.DataFrame:
    rows = []
    xls = pd.ExcelFile(xlsx_path)
    sheet_pattern = re.compile(r"^([A-Za-z]{2})\s*(\d{1,2})$")

    for sheet in xls.sheet_names:
        match = sheet_pattern.match(sheet.strip())
        if not match:
            continue
        river, point = match.groups()
        river = river.upper()
        point = int(point)
        df = standardize_demo_columns(pd.read_excel(xlsx_path, sheet_name=sheet))
        if "id" not in df.columns:
            continue
        df = df.copy()
        df["sheet"] = sheet
        df["river"] = river
        df["point"] = point
        df["fish_key"] = df["id"].map(lambda x: fish_key_from_demo_id(x, river=river, point=point))
        df["id_old_from_sheet"] = df["fish_key"].map(
            lambda x: f"{river} {point:02d} {int(str(x).split('_')[-1])}" if pd.notna(x) and "_" in str(x) else np.nan
        )
        keep = ["sheet", "river", "point", "fish_key", "id_old_from_sheet", "length_mm", "weight_g"]
        rows.append(df[[c for c in keep if c in df.columns]])

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


demo_all_raw = standardize_demo_columns(pd.read_excel(DEMO_XLSX, sheet_name="all"))
demo_all = demo_all_raw.copy()
demo_all["fish_key"] = demo_all["id"].map(fish_key_from_demo_id)

required_demo_cols = {"fish_key", "length_mm", "weight_g"}
missing_demo_cols = required_demo_cols - set(demo_all.columns)
if missing_demo_cols:
    raise ValueError(f"demo.xlsx all sheet is missing columns: {sorted(missing_demo_cols)}")

print("demo_all shape:", demo_all.shape)
display(demo_all.head())

individual_demo = build_demo_from_individual_sheets(DEMO_XLSX)
print("individual sheet combined shape:", individual_demo.shape)
display(individual_demo.head())

In [ ]:
if not individual_demo.empty:
    compare_cols = ["fish_key", "length_mm", "weight_g"]
    all_compare = demo_all[compare_cols].copy()
    sheet_compare = individual_demo[compare_cols].copy()

    all_counts = all_compare.value_counts(dropna=False).rename("all_count").reset_index()
    sheet_counts = sheet_compare.value_counts(dropna=False).rename("sheet_count").reset_index()
    demo_sheet_audit = all_counts.merge(sheet_counts, on=compare_cols, how="outer").fillna(0)
    demo_sheet_audit["count_diff"] = demo_sheet_audit["all_count"] - demo_sheet_audit["sheet_count"]
    demo_sheet_mismatches = demo_sheet_audit[demo_sheet_audit["count_diff"] != 0].copy()

    print("Rows in demo all:", len(all_compare))
    print("Rows in combined individual sheets:", len(sheet_compare))
    print("Value-level mismatches:", len(demo_sheet_mismatches))
    display(demo_sheet_mismatches.head(30))

    demo_sheet_mismatches.to_csv(OUTPUT_DIR / "demo_all_vs_individual_sheet_mismatches.csv", index=False)
else:
    print("No individual demo sheets were parsed.")

## 3. Image Index

Image filenames are expected to look like `cu1_001_01.png`: river, point, fish id, and image index.

In [ ]:
def build_image_index(root: Path, folders: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    pattern = re.compile(r"^([a-z]+)(\d+)_(\d+)_(\d+)\.png$", re.IGNORECASE)
    rows = []
    bad = []

    image_paths = []
    for folder in folders:
        folder_path = root / folder
        if folder_path.exists():
            image_paths.extend(p for p in folder_path.rglob("*.png") if p.is_file() and not p.name.startswith("._"))

    for path in sorted(image_paths):
        match = pattern.match(path.name)
        if not match:
            bad.append({"path": str(path), "file": path.name, "reason": "filename_pattern_mismatch"})
            continue

        river, point, fish_id, image_idx = match.groups()
        river = river.lower()
        point = int(point)
        fish_id = int(fish_id)
        image_idx = int(image_idx)
        fish_key = f"{river}{point}_{fish_id:03d}"
        scale_id = f"{fish_key}_{image_idx:02d}"
        rows.append({
            "path": str(path),
            "file": path.name,
            "scale_id": scale_id,
            "fish_key": fish_key,
            "river": river.upper(),
            "point": point,
            "fish_id": fish_id,
            "image_idx": image_idx,
            "relative_path": str(path.relative_to(root)),
        })

    images = pd.DataFrame(rows)
    bad_files = pd.DataFrame(bad)
    return images, bad_files


images_df, bad_image_files = build_image_index(ROOT_DIR, IMAGE_FOLDERS)

print(f"Parsed images: {len(images_df):,}")
print(f"Bad image filenames: {len(bad_image_files):,}")
if not images_df.empty:
    print("Images by river:")
    display(images_df["river"].value_counts().sort_index())
    display(images_df.head())
if not bad_image_files.empty:
    display(bad_image_files.head(20))
    bad_image_files.to_csv(OUTPUT_DIR / "bad_image_filenames.csv", index=False)

## 4. Labels and Master Table

`label` is derived from `age_est` and `obs`:
- numeric `age_est` becomes age class `0` to `5`;
- `not readable`, `regenerated`, or `broken` becomes class `6`;
- otherwise it stays unlabeled.

For later experiments, `age4` collapses readable ages into `0`, `1`, `2`, and `3+`.

In [ ]:
def clean_label(age_est, obs):
    if pd.notna(age_est):
        numeric = pd.to_numeric(age_est, errors="coerce")
        if pd.notna(numeric):
            return int(numeric)

    text = " ".join(
        normalize_text(x).lower()
        for x in [age_est, obs]
        if pd.notna(x)
    )
    if any(term in text for term in ["not readable", "regenerated", "broken"]):
        return 6
    return np.nan


def collapse_age4(label):
    if pd.isna(label) or int(label) == 6:
        return np.nan
    return min(int(label), 3)


labels_df = pd.read_excel(LABEL_XLSX, sheet_name="mizzou_allscales")
required_label_cols = {"scale_id", "age_est", "obs"}
missing_label_cols = required_label_cols - set(labels_df.columns)
if missing_label_cols:
    raise ValueError(f"labeling.xlsx is missing columns: {sorted(missing_label_cols)}")

labels_df = labels_df.copy()
labels_df["scale_id"] = labels_df["scale_id"].map(normalize_key)
labels_df["label"] = labels_df.apply(lambda r: clean_label(r["age_est"], r["obs"]), axis=1)
labels_df["label_source"] = np.where(labels_df["label"].notna(), "expert", "unlabeled")
labels_df["known_bad"] = np.where(labels_df["label"].eq(6), 1, 0)
labels_df["age4"] = labels_df["label"].map(collapse_age4)

print("labels_df shape:", labels_df.shape)
print("label counts:")
display(labels_df["label"].value_counts(dropna=False).sort_index())
display(labels_df.head())

In [ ]:
demo_keep_cols = ["fish_key", "id_old", "length_mm", "weight_g"]
demo_keep_cols = [c for c in demo_keep_cols if c in demo_all.columns]

label_keep_cols = ["scale_id", "sampling_date", "age_est", "obs", "label", "label_source", "known_bad", "age4"]
label_keep_cols = [c for c in label_keep_cols if c in labels_df.columns]

master_df = (
    images_df
    .merge(labels_df[label_keep_cols], on="scale_id", how="left")
    .merge(demo_all[demo_keep_cols], on="fish_key", how="left")
)

master_df["split"] = np.where(master_df["label"].notna(), "labeled", "unlabeled")
master_df["is_readable_labeled"] = master_df["label"].between(0, 5, inclusive="both")
master_df["is_age4_labeled"] = master_df["age4"].notna()

print("master_df shape:", master_df.shape)
print("split counts:")
display(master_df["split"].value_counts(dropna=False))
print("label counts:")
display(master_df["label"].value_counts(dropna=False).sort_index())
print("age4 counts, readable only:")
display(master_df["age4"].value_counts(dropna=False).sort_index())
display(master_df.head())

## 5. Data Audit

These checks are intentionally before any modeling. Resolve major mismatches or document why they are acceptable before training.

In [ ]:
def audit_master(master: pd.DataFrame, labels: pd.DataFrame, demo: pd.DataFrame) -> dict:
    duplicate_scale_ids = labels[labels["scale_id"].duplicated(keep=False)].sort_values("scale_id")
    images_without_labels = master[master["age_est"].isna() & master["obs"].isna()].copy()
    images_without_demo = master[master["length_mm"].isna() | master["weight_g"].isna()].copy()
    demo_without_images = demo.loc[~demo["fish_key"].isin(master["fish_key"])].copy()

    numeric_labeled = master[master["label"].between(0, 5, inclusive="both")].copy()
    label_sets = (
        numeric_labeled
        .groupby(["river", "point", "fish_id", "fish_key"])["label"]
        .agg(lambda s: sorted(set(int(x) for x in s.dropna())))
        .reset_index(name="labels")
    )
    fish_age_conflicts = label_sets[label_sets["labels"].map(len) > 1].copy()

    summary = {
        "images": len(master),
        "labeled_rows": int(master["label"].notna().sum()),
        "unlabeled_rows": int(master["label"].isna().sum()),
        "readable_labeled_rows": int(master["is_readable_labeled"].sum()),
        "known_bad_rows": int(master["known_bad"].fillna(0).sum()),
        "duplicate_scale_id_rows": len(duplicate_scale_ids),
        "images_without_label_rows": len(images_without_labels),
        "images_without_demo_rows": len(images_without_demo),
        "demo_fish_without_images": len(demo_without_images),
        "fish_age_conflict_groups": len(fish_age_conflicts),
    }
    return summary, duplicate_scale_ids, images_without_labels, images_without_demo, demo_without_images, fish_age_conflicts


audit_summary, duplicate_scale_ids, images_without_labels, images_without_demo, demo_without_images, fish_age_conflicts = audit_master(master_df, labels_df, demo_all)
audit_summary

In [ ]:
print("images_without_demo:")
display(images_without_demo[["scale_id", "fish_key", "river", "point", "fish_id", "image_idx", "path"]].head(50))

print("demo_fish_without_images:")
demo_cols_to_show = [c for c in ["fish_key", "id_old", "length_mm", "weight_g"] if c in demo_without_images.columns]
display(demo_without_images[demo_cols_to_show].head(50))

print("fish_age_conflicts:")
display(fish_age_conflicts.head(50))

print("duplicate_scale_ids:")
display(duplicate_scale_ids.head(50))

## 6. Save Clean Outputs

These CSVs can be used by later training notebooks so the merge logic stays consistent.

In [ ]:
master_path = OUTPUT_DIR / "master_table_new.csv"
summary_path = OUTPUT_DIR / "data_audit_summary.csv"
label_counts_path = OUTPUT_DIR / "label_counts.csv"
age4_counts_path = OUTPUT_DIR / "age4_counts.csv"

master_df.to_csv(master_path, index=False)
pd.DataFrame([audit_summary]).to_csv(summary_path, index=False)
master_df["label"].value_counts(dropna=False).sort_index().rename_axis("label").reset_index(name="count").to_csv(label_counts_path, index=False)
master_df["age4"].value_counts(dropna=False).sort_index().rename_axis("age4").reset_index(name="count").to_csv(age4_counts_path, index=False)

images_without_labels.to_csv(OUTPUT_DIR / "images_without_labels.csv", index=False)
images_without_demo.to_csv(OUTPUT_DIR / "images_without_demo.csv", index=False)
demo_without_images.to_csv(OUTPUT_DIR / "demo_fish_without_images.csv", index=False)
fish_age_conflicts.to_csv(OUTPUT_DIR / "fish_age_conflicts.csv", index=False)
duplicate_scale_ids.to_csv(OUTPUT_DIR / "duplicate_scale_ids.csv", index=False)

print("Saved:")
for path in [master_path, summary_path, label_counts_path, age4_counts_path]:
    print("-", path)
print("Output directory:", OUTPUT_DIR)

## 7. Notes for Next Notebook

Use `eda_outputs/master_table_new.csv` as the input table for modeling.

Recommended next modeling labels:
- `known_bad`: binary readable-quality model, where `1` means bad/broken/not-readable/regenerated.
- `age4`: readable-only age model, where `0`, `1`, `2`, and `3` mean `0+`, `1+`, `2+`, and `3+`.

Before modeling, confirm that `fish_age_conflicts.csv` is either empty or scientifically acceptable.